# 2 — Patterns

Loads the tables built by **notebook 1** and looks for structure. **It reads no logs** — if this
notebook runs, the dataset is genuinely self-contained.

The order is deliberate:

1. **What is in the dataset** — sessions per mouse per world, how long they run, how many trials.
2. **Each task on its own, per animal** — banish_multiplier and timeout_multiplier as two
   separate sections (never pooled); per animal first, then all animals grouped side by side.
3. **Per-session lines** — is there any shape over days?
4. **Blocks** — only now, and only if step 3 suggests something worth testing.
5. **First half vs second half.**

Steps 1–3 draw **no trend lines and no verdicts**. Picking a binning after seeing a curve is how a
pattern gets manufactured, so the un-binned answer comes first and `BLOCK_SIZE` is chosen in step 4,
after you have looked.

## Load ← YOU SET THIS

In [ ]:
MAIN_DIR = '/mnt/server/data'
PIPELINE_DIR = None
# =============================================================================

import sys, importlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

MAIN_DIR = Path(MAIN_DIR).expanduser()
cands = ([Path(PIPELINE_DIR)] if PIPELINE_DIR else []) + [
    Path.cwd().parent, Path.cwd(), Path.cwd().parent / 'session_pipeline']
PIPE = next((c.resolve() for c in cands if (c / 'common' / 'session_index.py').exists()), None)
sys.path.insert(0, str(PIPE / 'common')); sys.path.insert(0, str(PIPE / 'results'))

import session_index as sidx, perf_from_log as pfl, build_log_df as bl, plot_patterns as pp
sidx = importlib.reload(sidx); pfl = importlib.reload(pfl)
bl = importlib.reload(bl); pp = importlib.reload(pp)

# prefer the LOCAL copy saved by notebook 1, so this runs without the server mounted
_LOCAL = Path('~/repo/session_pipeline_output').expanduser()
OUT = _LOCAL if (_LOCAL / 'df_sessions.pkl').exists() else (MAIN_DIR / 'df_log')
print(f'loading dataset from {OUT}')
df_sessions = bl.load(OUT / 'df_sessions.pkl')
df_trials   = bl.load(OUT / 'df_trials.pkl')
print(f'\n{len(df_sessions)} sessions, {len(df_trials)} trials, '
      f'{df_sessions.mouse.nunique()} animal(s)')

## 1 — What is in the dataset

Before any performance question. A `D` curve cannot be read without knowing **how big each point is** — sessions vary a lot in length and trial count, and two points are only comparable if they carry a similar amount of data. *(For example, a long session of many trials sits on far firmer ground than a short one of a handful — the actual sizes for your sessions are what panels (d) session length and (e) trial count show; the numbers here are not assumed.)* Read those two panels first to see whether any two points can be compared at all.

In [ ]:
fig = pp.overview(df_sessions, df_trials)
plt.show()

display(df_sessions.groupby(['mouse', 'texture_id', 'task'])
        .agg(sessions=('session', 'count'), trials=('n_trials_total', 'sum'),
             minutes=('elapsed_min', 'sum'), first=('day', 'min'), last=('day', 'max'))
        .round(1))

## 2 — Choose what to look at ← YOU SET THIS

Chosen **after** the table above, not before. A world fixes the viewport, and therefore the chance
baseline, so sessions from different worlds are not directly comparable.

In [ ]:
ANIMAL = None        # None = every animal in the dataset, or e.g. 'JPAS_0168'
TEXTURE  = None        # None = every world, or e.g. 'W4'
TASK   = None        # None = every protocol, or e.g. 'banish_multiplier'
USABLE_ONLY = True   # drop sessions flagged unusable, and the earlier of any doubled day
# =============================================================================

X = df_sessions.copy()
if ANIMAL: X = X[X.mouse == ANIMAL]
if TEXTURE:  X = X[X.texture_id == TEXTURE]
if TASK:   X = X[X.task == TASK]
if USABLE_ONLY:
    X = X[X.use & X.keep_of_day & X.perf_error.eq('')]
X = X.sort_values(['day', 'time']).reset_index(drop=True)

T = df_trials[df_trials.session.isin(X.session)]
print(f'{len(X)} session(s), {len(T)} trial(s)   |   '
      f'{sorted(X.mouse.unique())}   {sorted(X.texture_id.unique())}   {sorted(X.task.unique())}')
if X.texture_id.nunique() > 1:
    print('\n  !! more than one TEXTURE selected: ' + ', '.join(sorted(X.texture_id.unique())))
    print('     A world fixes the viewport, so each has its own chance baseline. Set TEXTURE.')
if X.task.nunique() > 1:
    print('\n  !! more than one PROTOCOL selected: ' + ', '.join(sorted(X.task.unique())))
    for t in sorted(X.task.unique()):
        print(f'        {t:20s} {pfl.TASK_DESCRIPTION.get(t, "")}')
    print('     These are DIFFERENT EXPERIMENTS -- different icons, different reward units,')
    print('     different chance baseline. Step 3 will refuse to pool them. Set TASK.')
display(X[['session', 'day', 'task', 'n_trials', 'acc', 'chance', 'D', 'p', 'drops']].round(3))

## Extra — every per-session measure (reference tables)

The `D` tables above stay compact on purpose. This block surfaces **all** the columns notebook 1
builds for the selected sessions, grouped so nothing is hidden: the four chance baselines (each with
its own `D` and `p`), the **world-design opportunity** baseline, throughput/timing, the conflict
criterion, and the camera / switch flags.

Two things to keep straight here:

- **`good_opportunity_ratio` / `board_good_ratio` are a RATIO, not a p-value.** They are the board's
  good:bad icon SUPPLY (e.g. 2 reward : 1 punishment -> `0.667`). The p-value built from that supply is
  **`p_opportunity`** (session level). So the ratio you gave me feeds the test; it is not the test.
- The last table reads the **conflict / control criterion next to `p_opportunity`** for each session
  of the SAME task. (The earlier "board opportunity by world" table is dropped — within one
  banish_multiplier session the normal board is always `0.667` and the shadow-realm board is `NaN`, so
  splitting it by world compared `0.667` against nothing.)

Nothing is fitted here — these are tables to read.

In [ ]:
def _show(title, cols):
    have = [c for c in cols if c in X.columns]
    if not have:
        print(title, '\n  (none of these columns are present)\n'); return
    print(title)
    display(X[['session', 'day', 'task'] + have].round(3))

_show('BASELINES - observed accuracy vs the four chance levels (each with its D and p). '
      '\n  The opportunity ratio uses the world DESIGN (good:bad icon counts), NOT the viewport:',
      ['acc', 'chance', 'D', 'D_lo', 'D_hi', 'p',
       'chance_mem', 'D_mem', 'p_mem', 'chance_exo', 'D_exo', 'p_exo',
       'active_benefits', 'active_detriments', 'good_opportunity_ratio', 'p_opportunity'])

_show('THROUGHPUT & TIMING (elapsed = active + freeze):',
      ['pos', 'neg', 'drops', 'coll_per_min', 'drops_per_min', 'coll_per_active_min',
       'elapsed_min', 'active_min', 'freeze_min'])

_show('CONFLICT CRITERION (both types on screen; conflict = punishment was nearer):',
      ['n_both', 'n_conflict', 'k_conflict', 'conflict_p', 'conflict_lo', 'conflict_hi',
       'n_agree', 'agree_p'])

_show('CAMERA / SWITCH / FLAGS (camera_* is None until the video pipeline has run):',
      ['camera_stable', 'camera_move_frame', 'camera_move_ms',
       'switch_ms', 'n_before_switch', 'use', 'keep_of_day', 'note', 'warn'])

# --- observed accuracy vs EACH chance baseline: ONE SUBPLOT PER BASELINE (easier to read) ------
base = {'chance': ('#2980b9', 'zero-memory chance'),
        'chance_mem': ('#8e44ad', 'perfect-memory chance'),
        'chance_exo': ('#16a085', 'exogenous (nearest at spawn)'),
        'good_opportunity_ratio': ('#e67e22', 'world-design opportunity ratio')}
have = [c for c in base if c in X.columns]
if len(X) and have:
    fig, axs = plt.subplots(1, len(have), figsize=(4.2 * len(have), 4), sharey=True)
    axs = np.atleast_1d(axs); xi = np.arange(len(X))
    for ax, c in zip(axs, have):
        col, lab = base[c]
        ax.plot(xi, X['acc'], 'o-', color='k', lw=2, zorder=5, label='observed')
        ax.plot(xi, X[c], 's--', color=col, alpha=.9, label=lab)
        ax.fill_between(xi, X['acc'], X[c], where=(X['acc'] >= X[c]),
                        color=col, alpha=.15, interpolate=True)
        ax.set_xticks(xi); ax.set_xticklabels(X['session'], rotation=90, fontsize=6)
        ax.set_ylim(0, 1); ax.grid(alpha=.2); ax.legend(fontsize=7, loc='lower right')
        ax.set_title(lab, fontsize=9, fontweight='bold')
    axs[0].set_ylabel('P(positive)')
    fig.suptitle('Observed accuracy vs each chance baseline   '
                 '(shaded = observed ABOVE that baseline = better than chance)',
                 fontweight='bold', fontsize=10)
    plt.tight_layout(); plt.show()

# --- CONFLICT / CONTROL criterion next to the world-design p, PER SESSION (one task) ----------
# conflict_p, agree_p are PROPORTIONS (P collected a reward on that trial kind); p_opportunity is
# the p-value vs the board good:bad supply. board_good_ratio (trial level) is the SUPPLY ratio, not p.
crit = [c for c in ['session', 'day', 'task', 'conflict_p', 'agree_p',
                    'good_opportunity_ratio', 'p_opportunity'] if c in X.columns]
if crit:
    print('\nCONFLICT / CONTROL criterion next to the world-design opportunity p (per session):')
    print('  conflict_p = P(reward | punishment was nearer)   agree_p = control (reward nearer)')
    print('  good_opportunity_ratio = board good:bad SUPPLY (a ratio ~0.667, NOT a p-value)')
    print('  p_opportunity = one-sided p that his positives beat that supply ratio')
    display(X[crit].round(3))

## 3 — Each task on its own, per animal

`banish_multiplier` and `timeout_multiplier` are **different experiments** — different punishment
(shadow realm vs freeze), different icons, different chance baseline — so they are **never pooled**.
Each gets its **own clear section** below, drawn from every usable session in the dataset (this does
NOT depend on the Section-2 selection).

**With more than one animal:** each animal is shown **on its own first** (its sessions pooled into one
answer), then **all animals side by side, still grouped per animal** — never merged into one number,
because `D` is a within-animal measure. Each section shows, per animal:

- **CONTROL vs CONFLICT** — CONFLICT = the punishment was nearer, so proximity must be overridden (the
  number that must rise with learning); CONTROL = the reward was nearer (easy). The two must SEPARATE.
- **D** — discrimination vs the visibility baseline (above 0 = better than chance).
- **P_board** — does his success on each trial kind beat the world's good:bad icon supply?

In [ ]:
# usable sessions across the WHOLE dataset (both tasks, every animal) -- independent of Section 2
U = df_sessions[df_sessions.use & df_sessions.keep_of_day & df_sessions.perf_error.eq('')]
U = U.sort_values(['day', 'time']).reset_index(drop=True)
print(f'{len(U)} usable sessions   '
      f'|   animals {sorted(U.mouse.unique())}   tasks {sorted(U.task.unique())}')
for t in sorted(U.task.unique()):
    print(f'   {t:20s} {U[U.task == t].mouse.nunique()} animal(s), {(U.task == t).sum()} session(s)')

### 3a — banish_multiplier

In [ ]:
A_bm = pp.task_criterion(U[U.task == 'banish_multiplier'], sidx, 'banish_multiplier')
if len(A_bm):
    display(A_bm[[c for c in ['mouse','n_sessions','control_p','conflict_p','D',
                              'good_opportunity_ratio','D_opportunity','p_opportunity']
                 if c in A_bm.columns]].round(3))

### 3b — timeout_multiplier

In [ ]:
A_tm = pp.task_criterion(U[U.task == 'timeout_multiplier'], sidx, 'timeout_multiplier')
if len(A_tm):
    display(A_tm[[c for c in ['mouse','n_sessions','control_p','conflict_p','D',
                              'good_opportunity_ratio','D_opportunity','p_opportunity']
                 if c in A_tm.columns]].round(3))

## 4 — Per-session pattern

One point per session, every measure, on a shared x-axis so a bump in `D` can be read against what
throughput, trial count and session length were doing the same day.

**Nothing is fitted and nothing is concluded.** A session with few scored trials has a wide interval
— check the `n` panel before believing any single point.

### How to read the per-session pattern (section 4)

Each point is **one session**. **Curves are drawn PER MOUSE** (mixing animals on one x-axis makes
the line jump between them), then an **across-mice average** (each mouse collapsed to its own mean, so
a prolific animal does not dominate). Within a mouse the six panels share an x-axis, so a bump in one
lines up with the others. Nothing here is fitted - read the shape, then test it with the blocks below.

**Three chance baselines can appear, each giving its own `D`.**  `D = (observed - chance) / (1 - chance)`
= the share of the room above chance that he captured. What counts as "chance" depends on how you model
what he could have known:

- **Visibility-weighted** (`D`, solid line) - chance = how often each icon TYPE was actually inside his
  SCREEN (the viewport), time-weighted. What he could see *right now*. It depends on where he drove, so
  it is partly **endogenous** (steering away from the punishment lowers how often it is on screen, which
  the baseline then absorbs).
- **Spawn geometry** (`D (spawn geometry)` = `D_exo`, dashed) - chance = at each trial's START, was the
  NEAREST icon positive? Fixed by the board *before* he moves, so his behaviour cannot move it: an
  **exogenous** cross-check.
- **World-design opportunity** (`D (world-design opportunity)` = `D_opportunity`, dotted) - chance = the
  board's own good:bad icon SUPPLY (e.g. 2 reward : 1 punishment -> `0.667`), fixed by how the world was
  DESIGNED, using NO viewport at all. It answers "did he collect positives more often than the board
  simply hands them out?" - the cleanest cross-check, since his behaviour cannot move it.

The three **bracket the truth**: visibility can be gamed by where he parks, spawn-geometry assumes he
chases the nearest icon, and world-design ignores the screen entirely. If all three agree, the read is
solid.

**What a positive vs negative `D` means (the line at `D = 0` is CHANCE):**

- `D > 0` (**above** the line) - he collected positives MORE than that baseline predicts -> he is
  discriminating / avoiding the punishment. `D = 1` is perfect.
- `D = 0` (**on** the line) - exactly at chance: he collects in proportion to what the baseline makes
  available, i.e. no evidence he tells the icons apart.
- `D < 0` (**below** the line) - WORSE than chance: he collected the punishment MORE often than the
  baseline predicts. With few trials this is usually just noise (the confidence band crosses 0); a
  *consistently* negative `D` would mean an anti-preference. So a `D (spawn geometry)` point dipping
  below the line on some sessions is normal - that day he did slightly worse than "just grab the nearest
  icon" would have done, which small samples produce by chance.

**Panel by panel:**
1. **DISCRIMINATION (`D`)** - the baselines' D. 0 = chance, 1 = perfect, band = 95% confidence interval.
2. **OBSERVED vs its chance level** - his accuracy against the chance line(s); the **gap between the two
   lines IS `D`**.
3. **THROUGHPUT** - *productivity per minute, separate from accuracy* (he can be accurate but slow, or
   fast but at chance). Two rates: **collections/min** = how OFTEN he collected a reward, and **reward
   drops/min** = how MUCH reward he earned. A *drop* is one delivery of the reward pump; a collection
   made on a streak of 3 pays 3 drops - so drops >= collections, and "drops earned" is what used to be
   called "paid".
4. **HOW MUCH DATA** - scored collections per session; a low point means that session's `D` has a wide
   interval, so do not over-read it.
5. **SESSION LENGTH** - elapsed minutes (= active + freeze).
6. **TOTAL REWARD DROPS** - the reward he earned that session.

The **plot is directly below**, and the cell after it puts the **stochastic (world-design) opportunity**
test next to `D` and the criterion, so you can watch that p-value move relative to the others.

In [ ]:
# PER MOUSE first -- one clean curve per animal (mixing mice on one x-axis makes the line jump
# between animals). Then the ACROSS-mice average, equal weight per mouse.
mice = sorted(X.mouse.unique())
for m in mice:
    Xm = X[X.mouse == m]
    fig = pp.session_lines(Xm, title=f'{m} - {len(Xm)} sessions (nothing fitted)')
    plt.show()

if len(mice) > 1:                      # all sessions, averaged ACROSS mice (each mouse -> its mean)
    fig = pp.mouse_summary(X, title='All mice: per-mouse means + average across mice')
    plt.show()
else:
    print('single animal in the selection -- the per-mouse curve above is the whole picture')

In [ ]:
# --- the STOCHASTIC opportunity test next to D and the criterion, PER SESSION ---
Xo = X.copy()
if 'good_opportunity_ratio' in Xo.columns:
    Xo['D_opportunity'] = ((Xo['acc'] - Xo['good_opportunity_ratio'])
                           / (1 - Xo['good_opportunity_ratio']))
cmp_cols = [c for c in ['session', 'day', 'conflict_p', 'agree_p', 'control_p', 'D', 'D_exo', 'D_opportunity',
                        'good_opportunity_ratio', 'p', 'p_exo', 'p_opportunity'] if c in Xo.columns]
display(Xo[cmp_cols].round(3))

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5)); xi = np.arange(len(Xo))
a = ax[0]                                   # the three discrimination scores
for col, sty, lab in [('D', 'o-', 'D (visibility)'), ('D_exo', 's--', 'D (spawn geometry)'),
                      ('D_opportunity', '^:', 'D (world-design opportunity)')]:
    if col in Xo: a.plot(xi, Xo[col], sty, label=lab)
a.axhline(0, color='k', lw=1); a.set_ylabel('D'); a.grid(alpha=.2); a.legend(fontsize=8)
a.set_xticks(xi); a.set_xticklabels(Xo['session'], rotation=90, fontsize=6)
a.set_title('DISCRIMINATION vs each baseline (0 = chance)', fontweight='bold', fontsize=10)

a = ax[1]                                   # significance of each test (small = beats that baseline)
for col, sty, lab in [('p', 'o-', 'p (visibility)'), ('p_exo', 's--', 'p (spawn geometry)'),
                      ('p_opportunity', '^:', 'p (world-design opportunity)')]:
    if col in Xo: a.plot(xi, Xo[col].clip(1e-4, 1), sty, label=lab)
a.axhline(0.05, color='r', ls=':', label='p = 0.05'); a.set_yscale('log')
a.set_ylabel('p-value (log)'); a.grid(alpha=.2, which='both'); a.legend(fontsize=8)
a.set_xticks(xi); a.set_xticklabels(Xo['session'], rotation=90, fontsize=6)
a.set_title('SIGNIFICANCE of each test (below the red line = beats that baseline)',
            fontweight='bold', fontsize=10)
plt.tight_layout(); plt.show()

## 5 — Blocks ← ONLY IF STEP 4 SUGGESTS SOMETHING

Pooling raises the trial count per point, which narrows the intervals — but the bin should be chosen
because of what step 4 showed, not before looking. One session carries only ~10-15 conflict trials
(±0.25), so a single session is uninformative on its own.

**Blocks are calendar weeks by default** (Mon-Sun), so a block is "the week of the 24th-28th" rather
than "sessions 6-10". A week with fewer than `FULL_WEEK` sessions is incomplete and **falls back to 5
consecutive sessions**, so sparse stretches are not left as tiny one-session blocks. Set `BLOCK_MODE`
to an integer to force fixed sessions-per-block instead.

**Reminder — what "world-design opportunity" is.** The world is DESIGNED with a fixed supply of good
vs bad icons — here 2 reward : 1 punishment, so `good_opportunity_ratio = 2/(2+1) = 0.667`. That is a
plain *chance level*: an animal grabbing icons blindly would collect ~66.7 % positives just because
that is the mix on the board. It uses **no viewport and no memory** — only the world's design — so his
behaviour cannot move it, which makes it the cleanest cross-check.
- **`D_opportunity`** = how far his accuracy sits above that 0.667 supply, on the same 0-1 `D` scale
  as the other baselines (0 = at the supply, 1 = perfect).
- **`p_opportunity`** = the one-sided chance that he beat the supply by luck. **`p < 0.05` = a real
  effect**, and in the third panel those blocks are drawn dark green with a ★ (there is no "below 0" —
  a p-value runs 0-1; the test is whether it is under 0.05).

The third panel plots **`D_opportunity` next to `D (visibility)` on one shared `D` axis** so the two
discrimination scores are directly comparable, with the ★ marking the blocks that are significant.

In [ ]:
# BLOCKS. Default = CALENDAR WEEK (Mon-Sun): each block is one training week, e.g. 2026-08-24..28.
# A week with < FULL_WEEK sessions is incomplete -> it falls back to 5 CONSECUTIVE sessions.
# Set BLOCK_MODE = 'week', or an INTEGER for fixed sessions-per-block regardless of the calendar.
BLOCK_MODE = 'week'
FULL_WEEK  = 5       # how many sessions count as a complete week
# =============================================================================

B = (sidx.week_blocks(X, full_week=FULL_WEEK, fallback_size=5) if BLOCK_MODE == 'week'
     else sidx.blocks_of(X, size=int(BLOCK_MODE)))
cols = ['block', 'label', 'n_sessions', 'conflict_p', 'conflict_lo', 'conflict_hi', 'control_p',
        'D', 'good_opportunity_ratio', 'D_opportunity', 'p_opportunity']
display(B[[c for c in cols if c in B.columns]].round(3))

fig, ax = plt.subplots(1, 3, figsize=(19, 5)); x = np.arange(len(B))
a = ax[0]                                       # the criterion
a.errorbar(x, B.conflict_p, yerr=[(B.conflict_p - B.conflict_lo).clip(lower=0),
                                  (B.conflict_hi - B.conflict_p).clip(lower=0)],
           fmt='o-', color='#e67e22', capsize=4, lw=2, label='CONFLICT (must rise)')
a.errorbar(x, B.control_p, yerr=[(B.control_p - B.control_lo).clip(lower=0),
                                 (B.control_hi - B.control_p).clip(lower=0)],
           fmt='s--', color='#7f8c8d', capsize=4, label='CONTROL (stays high)')
a.axhline(.5, ls=':', color='k'); a.set_ylim(0, 1)
a.set_xticks(x); a.set_xticklabels(B.label, fontsize=7, rotation=45, ha='right')
a.set_ylabel('P(collected the reward)'); a.legend(fontsize=8)
a.set_title('THE CRITERION, per block', fontweight='bold'); a.grid(alpha=.2)

a = ax[1]                                       # the gap
a.bar(x, B.control_p - B.conflict_p, color='#c0392b')
a.set_xticks(x); a.set_xticklabels(B.label, fontsize=7, rotation=45, ha='right')
a.set_ylabel('control - conflict')
a.set_title('THE GAP per block\n(shrinking = overriding proximity to avoid the punishment)',
            fontweight='bold'); a.grid(alpha=.2, axis='y')

a = ax[2]                                       # world-design opportunity D per block (ONE axis)
if 'D_opportunity' in B.columns:
    sig = B.p_opportunity < 0.05                          # significant = beats the good:bad supply
    a.bar(x, B.D_opportunity, color=['#16a085' if s else '#b5d8cf' for s in sig],
          label='D (world-design opportunity)')
    a.plot(x, B.D, 's--', color='#2c3e50', label='D (visibility)')   # same D scale -> directly comparable
    a.axhline(0, color='k', lw=1)                         # 0 = at the good:bad supply (chance)
    for i, s in enumerate(sig):                           # star the significant blocks
        if s:
            a.text(x[i], B.D_opportunity.iloc[i] + 0.01, '*', ha='center', va='bottom', fontsize=16)
    a.set_ylabel('D  (0 = at the good:bad supply)'); a.legend(fontsize=8, loc='upper left')
a.set_xticks(x); a.set_xticklabels(B.label, fontsize=7, rotation=45, ha='right')
a.set_title('WORLD-DESIGN opportunity per block\n'
            '(bar above 0 = beats the supply; * = significant, p_opportunity < 0.05)',
            fontweight='bold'); a.grid(alpha=.2, axis='y')
plt.tight_layout(); plt.show()

## 6 — First half vs second half

A **fixed** split, so it is not chosen by eye. Two different questions kept apart: **within session**
(does he fade during a session?) and **across days** (did he change over training?). Within-session
halves are pooled per session so one long session cannot dominate both sides. Escapes are excluded —
they offer no reward-vs-punishment choice.

The numbers and their intervals are printed; read them yourself.

In [ ]:
fig, HALVES, PERSESS = pp.half_split(T, X)
plt.show()

for k, d in HALVES.items():
    if 'within' in k:
        print(f"{k:15s} P(reward) = {d['p']:.3f}   n = {d['n']:4d}   "
              f"95% confidence interval [{d['lo']:.3f}, {d['hi']:.3f}]")
    else:
        print(f"{k:15s} mean D    = {d['D']:+.3f}   over {d['n']} session(s)")
print('\nOverlapping intervals mean the difference is not resolved by this much data.')
PERSESS